# **GPT**

Task 1: Data Preparation

Objective

Prepare text data for GPT training.

- Load dataset
- Convert to lowercase
- Remove unwanted symbols
- Tokenize text
- Build vocabulary
- Create input-output sequences

Example: Input: 'the cat chased' Target: 'the'

In [74]:
import re
from tensorflow.keras.preprocessing.text import Tokenizer

# Load Dataset
texts = [
    "The cat chased the mouse.",
    "The dog barked loudly.",
    "Machine learning is powerful.",
    "Deep learning uses neural networks."
]

# Lowercase + Remove Symbols
texts = [re.sub(r'[^a-zA-Z\s]', '', t.lower()) for t in texts]

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

word_index = tokenizer.word_index
vocab_size = len(word_index) + 1

# Create Input-Output Sequences
sequences = []

for text in texts:
    tokens = tokenizer.texts_to_sequences([text])[0]

    for i in range(1, len(tokens)):
        sequences.append((tokens[:i], tokens[i]))

print("Vocabulary Size:", vocab_size)
print("Sample Sequence:", sequences[0])

Vocabulary Size: 16
Sample Sequence: ([1], 3)


## ***TASK-2 Token Embeddings***

In [75]:
import tensorflow as tf

embedding_dim = 32

embedding_layer = tf.keras.layers.Embedding(
    vocab_size,
    embedding_dim
)

sample_input = tf.constant([[1, 2, 3]])

embedded = embedding_layer(sample_input)

print("Embeddings:")
print(embedded)

print("Embedding Shape:", embedded.shape)

Embeddings:
tf.Tensor(
[[[ 0.04394232 -0.0008554  -0.02083069 -0.02230573 -0.0498783
   -0.00669842 -0.00726644  0.03365364 -0.0160921   0.00852077
   -0.00556041 -0.04557982 -0.00577834  0.04131441  0.02971439
   -0.0257014  -0.04512984 -0.02741641 -0.00905963  0.01832484
   -0.04686743  0.02921769 -0.04625142 -0.00476038 -0.04814223
    0.02741635  0.04175839  0.00026028  0.03410149  0.04846216
    0.03109963 -0.02051621]
  [-0.0256663  -0.0163218   0.0006899   0.01193004  0.04340059
    0.03264498 -0.03554773 -0.00373399  0.00025426 -0.01994641
    0.04445908 -0.00830752  0.01578805  0.04927884 -0.00336589
   -0.047578   -0.03350201 -0.00276598  0.0207654  -0.04837225
    0.01140302  0.00183947 -0.04340969  0.04332482 -0.0004138
    0.04855808  0.03801893 -0.00912316  0.04014819 -0.01413244
    0.02665525 -0.03901826]
  [ 0.01148156  0.02515194  0.03283961 -0.02215091  0.02966039
    0.02443704 -0.00818459 -0.04706457 -0.00248909 -0.02186163
   -0.00970708 -0.00515424 -0.00043119  0

# ***Task 3: Positional Encoding***
Objective

Add positional information.

In [76]:
import numpy as np

def positional_encoding(max_len, d_model):

    pos = np.arange(max_len)[:, np.newaxis]
    i = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(
        10000,
        (2 * (i // 2)) / np.float32(d_model)
    )

    angles = pos * angle_rates

    pe = np.zeros((max_len, d_model))

    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])

    return pe

pe = positional_encoding(10, 32)

print(pe.shape)

(10, 32)


## ***Task 4: Masked Self Attention***

Objective

Implement causal attention.

In [77]:
import tensorflow as tf

seq_len = 5

mask = 1 - tf.linalg.band_part(
    tf.ones((seq_len, seq_len)),
    -1,
    0
)

print(mask.numpy())

[[0. 1. 1. 1. 1.]
 [0. 0. 1. 1. 1.]
 [0. 0. 0. 1. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0.]]


## ***Task 5: Multi Head Attention***

Implement:

4 Attention Heads

In [78]:
mha = tf.keras.layers.MultiHeadAttention(
    num_heads=4,
    key_dim=32
)

x = tf.random.normal((2, 5, 32))

attn_output = mha(
    x,
    x,
    attention_mask=mask
)

print("Attention Output Shape:")
print(attn_output.shape)

Attention Output Shape:
(2, 5, 32)


## ***Task-6 : GPT Decoder Block***
    Masked Multi Head Attention
        ↓
    Add & Normalize
        ↓
    Feed Forward Network
        ↓
    Add & Normalize
    
Implement from scratch.

In [79]:
import tensorflow as tf

class DecoderBlock(tf.keras.layers.Layer):

    def __init__(self, num_heads, d_model, dff):
        super().__init__()

        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model
        )

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(
                dff,
                activation="relu"
            ),
            tf.keras.layers.Dense(d_model)
        ])

        self.norm1 = tf.keras.layers.LayerNormalization()
        self.norm2 = tf.keras.layers.LayerNormalization()

    def call(self, x, attention_mask=None):

        attn_output = self.mha(
            query=x,
            value=x,
            key=x,
            attention_mask=attention_mask
        )

        x = self.norm1(x + attn_output)

        ffn_out = self.ffn(x)

        x = self.norm2(x + ffn_out)

        return x

## ***Task-7 : GPT Model***

Architecture:

    Input Tokens
      ↓
    Embedding
      ↓
    Position Encoding
      ↓
    Decoder Block
      ↓
    Decoder Block
      ↓
    Linear
      ↓
    Softmax

In [82]:
import tensorflow as tf

# GPT Model

inputs = tf.keras.Input(shape=(None,))

# Embedding Layer
x = tf.keras.layers.Embedding(
    vocab_size,
    32
)(inputs)

# Decoder Blocks
decoder = DecoderBlock(
    num_heads=4,
    d_model=32,
    dff=64
)

x = decoder(x)
x = decoder(x)

# Take Last Token Output
x = tf.keras.layers.Lambda(
    lambda t: t[:, -1, :]
)(x)

# Output Layer
outputs = tf.keras.layers.Dense(
    vocab_size,
    activation='softmax'
)(x)

# Create Model
model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs
)

model.summary()

Model: "functional_17"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_21      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_21        │ (None, None, 32)  │        512 │ input_layer_21[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_block_14    │ (None, None, 32)  │     21,120 │ embedding_21[0][… │
│ (DecoderBlock)      │                   │            │ decoder_block_14… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_9 (Lambda)   │ (None, 32)        │          0 │ decoder_block_14… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_38 (Dense)    │ (None, 16)        │        528 │ lambda_9[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 22,160 (86.56 KB)

 Trainable params: 22,160 (86.56 KB)

 Non-trainable params: 0 (0.00 B)

## *Task 8: Next Token Prediction*

In [83]:
import numpy as np
import re
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Prepare Training Data
X = []
y = []

for seq in sequences:
    X.append(seq[0])
    y.append(seq[1])

max_len = max(len(seq) for seq in X)

X = pad_sequences(
    X,
    maxlen=max_len,
    padding='pre'
)

y = tf.keras.utils.to_categorical(
    y,
    num_classes=vocab_size
)

print("X Shape:", X.shape)
print("y Shape:", y.shape)

# Compile Model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train Model
model.fit(
    X,
    y,
    epochs=100,
    verbose=1
)

# Reverse Dictionary
index_word = {
    v: k
    for k, v in tokenizer.word_index.items()
}

# Next Token Prediction Function
def predict_next_word(text):

    text = re.sub(
        r'[^a-zA-Z\s]',
        '',
        text.lower()
    )

    seq = tokenizer.texts_to_sequences([text])[0]

    seq = pad_sequences(
        [seq],
        maxlen=max_len,
        padding='pre'
    )

    pred = model.predict(
        seq,
        verbose=0
    )

    next_word_id = np.argmax(pred[0])

    return index_word.get(
        next_word_id,
        "Unknown"
    )

# Example
input_text = "the cat chased"

print("Input :", input_text)
print("Prediction :", predict_next_word(input_text))

X Shape: (14, 4)
y Shape: (14, 16)
Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.0000e+00 - loss: 3.5069
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.0714 - loss: 2.9374
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.2143 - loss: 2.5190
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.4286 - loss: 2.2059
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - accuracy: 0.4286 - loss: 1.9341
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.4286 - loss: 1.6861
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5000 - loss: 1.4634
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5714 - loss: 1.2748
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.6429 - loss: 1.1116
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.8571 - loss: 0.9740
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.8571 - loss: 0.8582
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━

## ***Task 9: Text Generation***

In [85]:
def generate_text(
    seed_text,
    num_words
):

    for _ in range(num_words):

        next_word = predict_next_word(
            seed_text
        )

        if next_word == "Unknown":
            break

        seed_text += " " + next_word

    return seed_text

# Example
generated_text = generate_text(
    "deep learning uses",
    2

)

print("Generated Text:")
print(generated_text)

Generated Text:
deep learning uses neural networks
